# PyTorch for LLMs 03: Data, Checkpoints, and Hyperparameters

> **What you finished last time:** a full-batch \`nn.Module\` saved to \`../02-models-and-training-loops/artifacts/part02_linear_state.pt\`.
> **What this notebook delivers:** manual and DataLoader batch evidence, reload parity, a resumable checkpoint, and a one-variable learning-rate comparison.
> **Prerequisite for the next notebook:** the distinction between best-model and resumable state.
> **Missing artifact behavior:** run Part 02 first; this notebook fails loudly rather than inventing replacement state.

**Guiding question:** Which state belongs to the model, which belongs to the experiment, and which belongs to the optimizer?

\`\`\`mermaid
flowchart LR
    A["Examples"] --> B["Manual index batches"]
    B --> C["Dataset contract"]
    C --> D["DataLoader policy"]
    D --> E["Training updates"]
    E --> F["Checkpoint"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Imports, model definition, and artifact paths ----------------------------
import json
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FuelLinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, inputs):
        return self.linear(inputs)

part02_dir = Path("../02-models-and-training-loops/artifacts")
state_path = part02_dir / "part02_linear_state.pt"
config_path = part02_dir / "part02_config.json"
if not state_path.exists() or not config_path.exists():
    raise FileNotFoundError("Part 02 artifacts are missing. Execute Part 02 before Part 03.")
print("Part 02 artifacts found; fresh-kernel reload can proceed.")


In [ ]:
# -- Reload inter-notebook state and verify reconstruction facts ---------------
config = json.loads(config_path.read_text(encoding="utf-8"))
reloaded_model = FuelLinearModel().to(device)
reloaded_model.load_state_dict(torch.load(state_path, map_location=device, weights_only=True))
reloaded_model.eval()
print(f"Reloaded keys: {list(reloaded_model.state_dict())}")
print(f"Configuration: {config}")


## 1. Attempt 1: make batches by hand

Full-batch training hides the policy for ordering and grouping examples. Start with an explicit permutation.

**Predict:** Does shuffling change which examples exist? No. It changes which examples share an update and their order.


In [ ]:
# -- Recreate the deterministic data contract from Part 02 -------------------
torch.manual_seed(config["seed"])
n_examples = config["n_examples"]
engine_load = torch.linspace(0.1, 1.0, n_examples)
ambient_temp = torch.linspace(-0.8, 0.9, n_examples).roll(17)
X = torch.stack([engine_load, ambient_temp], dim=1)
noise = 0.03 * torch.randn(n_examples, 1)
y = 1.8 * X[:, :1] - 0.65 * X[:, 1:2] + 0.4 + noise
print(f"Recreated X={tuple(X.shape)}, y={tuple(y.shape)} from saved configuration.")


In [ ]:
# -- Build shuffled manual batches and expose the remainder policy -------------
batch_size = 20
generator = torch.Generator().manual_seed(31)
permutation = torch.randperm(n_examples, generator=generator)
manual_batches = [
    permutation[start : start + batch_size]
    for start in range(0, n_examples, batch_size)
]
covered = torch.cat(manual_batches).sort().values
print(f"Batch sizes: {[len(indices) for indices in manual_batches]}")
print(f"All examples covered once: {torch.equal(covered, torch.arange(n_examples))}")
print("The smaller final batch is retained deliberately.")


### That worked, but indexing policy is mixed with training code

\`Dataset\` answers "what is example i?" \`DataLoader\` answers "in what order and groups should examples arrive?"


In [ ]:
# -- Express example and batch contracts with Dataset and DataLoader ----------
dataset = TensorDataset(X, y)
loader_generator = torch.Generator().manual_seed(31)
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=loader_generator,
    drop_last=False,
)
first_X, first_y = next(iter(loader))
print(f"One example: X{tuple(dataset[0][0].shape)}, y{tuple(dataset[0][1].shape)}")
print(f"One batch: X{tuple(first_X.shape)}, y{tuple(first_y.shape)}")


In [ ]:
# -- Prove the DataLoader covers the same examples without dropping -----------
ordered_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
loaded_X = torch.cat([batch_X for batch_X, _ in ordered_loader])
loaded_y = torch.cat([batch_y for _, batch_y in ordered_loader])
print(f"Feature parity: {torch.equal(loaded_X, X)}")
print(f"Target parity: {torch.equal(loaded_y, y)}")
print(f"Last batch size: {len(list(ordered_loader)[-1][0])}")


## 2. A state dictionary is necessary but not sufficient

\`\`\`text
model state:      learned tensors
optimizer state:  momentum / adaptive statistics
experiment config: architecture and policy facts
progress state:   epoch and best metric
\`\`\`

\`\`\`mermaid
flowchart LR
    A["state_dict"] --> D["Inference reload"]
    A --> E["Resumable checkpoint"]
    B["optimizer state"] --> E
    C["epoch + config"] --> E
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
\`\`\`


In [ ]:
# -- Prove save/load prediction parity ----------------------------------------
probe = X[:7].to(device)
with torch.inference_mode():
    before_reload = reloaded_model(probe)
parity_model = FuelLinearModel().to(device)
parity_model.load_state_dict(torch.load(state_path, map_location=device, weights_only=True))
parity_model.eval()
with torch.inference_mode():
    after_reload = parity_model(probe)
max_reload_gap = (before_reload - after_reload).abs().max().item()
print(f"Maximum reload prediction gap: {max_reload_gap:.2e}")
assert max_reload_gap == 0.0


In [ ]:
# -- Train mini-batches and save resumable state ------------------------------
train_model = FuelLinearModel().to(device)
optimizer = torch.optim.Adam(train_model.parameters(), lr=0.03)
loss_function = nn.MSELoss()
loader_generator = torch.Generator().manual_seed(31)
train_loader = DataLoader(dataset, batch_size=20, shuffle=True, generator=loader_generator)
for epoch in range(25):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = loss_function(train_model(batch_X), batch_y)
        loss.backward()
        optimizer.step()

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)
resume_path = artifact_dir / "part03_resumable.pt"
torch.save({
    "epoch": epoch,
    "model_state": train_model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "config": {"batch_size": 20, "learning_rate": 0.03},
}, resume_path)
print(f"Saved resumable checkpoint at epoch {epoch + 1}: {resume_path}")


### Code walkthrough: resumable checkpoint

- \`model_state\` reproduces the learned function.
- \`optimizer_state\` reproduces Adam's accumulated update statistics.
- \`epoch\` locates progress.
- \`config\` explains the batching and learning-rate policy.
- A best-model checkpoint can be smaller because inference does not need optimizer progress.


## 3. Hyperparameters are controlled comparisons

**Predict:** Is a larger learning rate always faster? It may reduce loss faster, overshoot, or diverge. Change one variable and record the same number of updates.


In [ ]:
# -- Compare one variable: learning rate --------------------------------------
def loss_after_updates(learning_rate, updates=30):
    torch.manual_seed(41)
    candidate = FuelLinearModel().to(device)
    candidate_optimizer = torch.optim.SGD(candidate.parameters(), lr=learning_rate)
    local_X, local_y = X.to(device), y.to(device)
    losses = []
    for _ in range(updates):
        candidate_optimizer.zero_grad()
        candidate_loss = loss_function(candidate(local_X), local_y)
        candidate_loss.backward()
        candidate_optimizer.step()
        losses.append(candidate_loss.item())
    return losses

for learning_rate in (0.005, 0.05, 0.5):
    losses = loss_after_updates(learning_rate)
    print(f"lr={learning_rate:5.3f}: {losses[0]:.4f} -> {losses[-1]:.4f}")


### Your turn: change only the batch size


In [ ]:
# -- CHANGE THIS: inspect the resulting batch contract ------------------------
YOUR_BATCH_SIZE = 16
your_loader = DataLoader(dataset, batch_size=YOUR_BATCH_SIZE, shuffle=False, drop_last=False)
your_sizes = [len(batch_X) for batch_X, _ in your_loader]
print(f"Batch sizes at {YOUR_BATCH_SIZE}: {your_sizes}")
assert sum(your_sizes) == len(dataset)


## Scorecard and coverage ledger

| Tier | Covered here |
|---|---|
| Built and measured | manual batches, shuffle/coverage contract, Dataset, DataLoader, state reload parity, resumable checkpoint, learning-rate comparison |
| Explained or illustrated | model state versus optimizer/config/progress state |
| Named and out of scope | distributed sampling, worker tuning, gradient accumulation across batches, checkpoint retention policy |

**Common failure modes:** accidental \`drop_last\`, nondeterministic comparisons, loading weights into the wrong architecture, and calling a weight-only file "resumable."

**Next:** Part 04 separates train and validation lifecycles, adds mode changes, scheduling, early stopping, and two checkpoint purposes.
